# Übungen: Recommender Systeme

Versuche jede Aufgabe zuerst selbst zu lösen — die Musterlösung kommt am Ende jeder Zelle.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings('ignore')

---
## Aufgabe 1 — Cosine Similarity verstehen

Gegeben sind drei Buch-Vektoren (Genre-Features als One-Hot):

| Buch | Krimi | Fantasy | Sci-Fi | Romanze |
|------|-------|---------|--------|---------|
| Buch A | 1 | 0 | 1 | 0 |
| Buch B | 1 | 0 | 1 | 0 |
| Buch C | 0 | 1 | 0 | 1 |

**Frage:** Welche zwei Bücher sind sich am ähnlichsten? Berechne die Cosine Similarity.

In [ ]:
# Deine Lösung:
books = pd.DataFrame({
    'Krimi':   [1, 1, 0],
    'Fantasy':  [0, 0, 1],
    'Sci-Fi':   [1, 1, 0],
    'Romanze':  [0, 0, 1]
}, index=['Buch A', 'Buch B', 'Buch C'])

# Berechne hier die Cosine Similarity Matrix:
# sim = ...
# print(sim)

<details>
<summary>Musterlösung</summary>

```python
sim = cosine_similarity(books)
sim_df = pd.DataFrame(sim, index=books.index, columns=books.index)
print(sim_df.round(3))
# Buch A und Buch B: Similarity = 1.0 (identisch)
# Buch A/B und Buch C: Similarity = 0.0 (völlig verschieden)
```
</details>

---
## Aufgabe 2 — Content-Based Recommender bauen

Baue einen einfachen Buch-Recommender. Der Datensatz hat 6 Bücher mit Genre und Autor.

In [ ]:
buecher = pd.DataFrame({
    'titel':  ['Dune', 'Foundation', '1984', 'Brave New World',
               'The Hobbit', 'Lord of the Rings'],
    'genre':  ['Sci-Fi', 'Sci-Fi', 'Dystopie', 'Dystopie', 'Fantasy', 'Fantasy'],
    'autor':  ['Herbert', 'Asimov', 'Orwell', 'Huxley', 'Tolkien', 'Tolkien'],
    'seiten': [412, 255, 328, 311, 310, 1200]
})

# Aufgabe:
# 1. Features mit pd.get_dummies erstellen (genre + autor)
# 2. 'seiten' skalieren mit StandardScaler
# 3. Cosine Similarity berechnen
# 4. Funktion schreiben die für ein Buch die 2 ähnlichsten zurückgibt
# 5. Teste mit 'Dune'

# Deine Lösung:


<details>
<summary>Musterlösung</summary>

```python
features = pd.get_dummies(buecher[['genre', 'autor']])
scaler = StandardScaler()
features['seiten'] = scaler.fit_transform(buecher[['seiten']])

sim = cosine_similarity(features)

def empfehle(titel, n=2):
    idx = buecher.index[buecher['titel'] == titel][0]
    scores = sorted(enumerate(sim[idx]), key=lambda x: x[1], reverse=True)
    top = [i[0] for i in scores[1:n+1]]
    return buecher['titel'].iloc[top].values

print(empfehle('Dune'))   # ['Foundation'] wegen gleichem Genre
```
</details>

---
## Aufgabe 3 — User-Item-Matrix & Rating-Zentrierung

Nutzer 1-4 haben Bücher bewertet. Erstelle die User-Item-Matrix und zentriere die Ratings.

In [ ]:
ratings = pd.DataFrame({
    'userId': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4],
    'buch':   ['Dune', 'Foundation', '1984',
               'Dune', '1984',
               'The Hobbit', 'Lord of the Rings', 'Foundation',
               'Brave New World', 'The Hobbit'],
    'rating': [5, 4, 2, 4, 3, 5, 5, 3, 4, 3]
})

# Aufgabe:
# 1. User-Item-Matrix erstellen (userId als Index, buch als Spalten, fill_value=0)
# 2. Eine neue Spalte 'mean_rating' berechnen (Durchschnitt pro User)
# 3. Eine neue Spalte 'centered_rating' berechnen (rating - mean_rating)
# 4. Was fällt dir auf? Welcher Nutzer ist der "strengste Bewerter"?

# Deine Lösung:


<details>
<summary>Musterlösung</summary>

```python
user_item = ratings.pivot_table(index='userId', columns='buch',
                                 values='rating', fill_value=0)
print(user_item)

ratings['mean_rating']     = ratings.groupby('userId')['rating'].transform('mean')
ratings['centered_rating'] = ratings['rating'] - ratings['mean_rating']

print(ratings.groupby('userId')['mean_rating'].first())
# Nutzer 1 hat mean=3.67, Nutzer 3 hat mean=4.33 → Nutzer 3 gibt generell höhere Noten
# Zentrierung macht Ratings vergleichbar!
```
</details>

---
## Aufgabe 4 — Kaltstartproblem

**Konzeptfrage** — keine Code-Aufgabe.

Ein neuer Nutzer registriert sich bei einem Streaming-Dienst und hat noch nichts geschaut.

1. Warum kann **Collaborative Filtering** diesem Nutzer nichts empfehlen?
2. Warum kann **Content-Based Filtering** diesem Nutzer auch nichts empfehlen?
3. Was machen Netflix/Spotify stattdessen? (Denke an den Onboarding-Prozess)

<details>
<summary>Musterlösung</summary>

1. **Collaborative Filtering** braucht eine User-History um ähnliche Nutzer zu finden. Ohne Ratings gibt es nichts zu vergleichen.

2. **Content-Based Filtering** braucht ein Nutzerprofil (welche Genres/Schauspieler mag er?). Ohne Interaktionen gibt es kein Profil.

3. **Lösungen in der Praxis:**
   - Onboarding: Nutzer wählt explizit Lieblings-Genres/Artists
   - Popularity-Based: Einfach die beliebtesten Items empfehlen
   - Demografische Daten nutzen (Alter, Region)
   - Hybrid: Content-Based bis genug History vorhanden, dann wechseln
</details>

---
## Bonus — Surprise mit echten MovieLens-Daten

Wenn du das Notebook `week_29_ml/01_recommenders/` bereits durchgearbeitet hast:

In [ ]:
# Aufgabe: Lade die echten MovieLens-Daten und trainiere einen SVD-Recommender
# Daten liegen in: week_29_ml/01_recommenders/data/movielens-small/

# 1. movies.csv und ratings.csv einlesen
# 2. Surprise Dataset erstellen
# 3. SVD trainieren
# 4. RMSE ausgeben
# 5. Vorhersage für userId=1, movieId=1 ausgeben

# Deine Lösung:


<details>
<summary>Musterlösung</summary>

```python
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

ratings = pd.read_csv('../../week_29_ml/01_recommenders/data/movielens-small/ratings.csv')

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
model = SVD(random_state=42)
model.fit(trainset)

predictions = model.test(testset)
accuracy.rmse(predictions)

pred = model.predict(uid=1, iid=1)
print(f"Vorhergesagtes Rating: {pred.est:.2f}")
```
</details>